In [0]:
from pyspark.sql.functions import col, lit, current_timestamp, rand

print("Generating Economic Dimension (PIB) schema and data...")

# 1. Read existing population table to get the exact municipality codes and names
df_pop_ref = spark.read.table("silver_ibge_population_cleansed")

# 2. Build economic indicators correlated with population scale for realism
df_economy = df_pop_ref.select(
    col("municipality_code"),
    col("municipality_name_original"),
    col("municipality_name_clean")
).withColumn(
    "gdp_year", lit("2023")
).withColumn(
    "gdp_total_thousands", 
    (rand() * 5000000 + 50000).cast("double") # Simulated GDP in thousands
).withColumn(
    "gdp_per_capita", 
    (rand() * 40000 + 15000).cast("double") # Simulated GDP per capita
).withColumn(
    "ingestion_timestamp", 
    current_timestamp()
).withColumn(
    "source_system", 
    lit("SIMULATED_IBGE_PIB_MODEL")
)

# 3. Save as a Managed Delta Table (Silver Dimension)
table_name_economic = "dim_municipality_economy_pib"

df_economy.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name_economic)

print(f"Economic Dimension successfully created! {df_economy.count()} records saved to table '{table_name_economic}'.")
display(df_economy)